# COMP662 Assignment 1 — Synthetic Dataset Generation


In [ ]:
# Import the existing assignment implementation and the small helpers used below.
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

# Use paths relative to this notebook so it runs from notebooks/ or the repository root.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import assignment1 as a1
DATA_PATH = ROOT / "data" / "train.csv"
FIGURES = ROOT / "figures"
MODELS = ROOT / "models"
SEED = 42

a1.set_seed()
FIGURES.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)


## Task 1 — Data understanding and concise exploratory data analysis


### 1.1 Class distribution: counts and percentages


The class table and chart below show both counts and percentages. The same class proportions are preserved later with stratified splitting.


In [ ]:
# Load the released training data and separate the target from its ten numerical features.
data = pd.read_csv(DATA_PATH)
target = "Class"
features = [column for column in data.columns if column != target]
assert data.shape == (12_111, 11)
assert data[target].between(0, 4).all()

# Count each bean class and calculate its percentage of the full dataset.
class_distribution = data[target].value_counts().sort_index().rename("count").to_frame()
class_distribution["percentage"] = 100 * class_distribution["count"] / len(data)
display(class_distribution)

# Reuse the assignment plotting function to save the class-distribution figure.
a1.save_eda(data, features)
display(Image(filename=FIGURES / "class_distribution.png"))


### 1.2 Visual analysis of representative features


Area, Perimeter, MajorAxisLength, and Eccentricity are representative size and shape features. The plot compares their distributions across all five classes.


In [ ]:
# Display the four-feature distribution figure generated from the real training data.
display(Image(filename=FIGURES / "feature_distributions.png"))


### 1.3 Continuous-feature correlation analysis


The heatmap checks correlations among all continuous features. It helps identify related size measurements and supports the later choice of a non-linear classifier.


In [ ]:
# Display the correlation heatmap calculated from the ten numerical features.
display(Image(filename=FIGURES / "real_correlation.png"))


### 1.4 EDA effects on preprocessing, model, metrics, and cross-validation


The data has no missing values, so no imputation is used. Class imbalance makes macro-F1 the primary metric and accuracy the secondary metric. Stratified splitting and stratified five-fold CV preserve class proportions.


## Task 2 — Baseline classifier development


### 2.1 Necessary preprocessing and justification


The random-forest baseline uses the original numerical features because tree splits do not require scaling. Scaling is used only for the VAE, where gradient-based training benefits from comparable feature ranges.


### 2.2 Baseline classifier selection and justification


I use a class-weighted random forest. It can model non-linear feature relationships and its class weighting gives minority classes more influence during training.


### 2.3 Primary metric, secondary metric, and cross-validation strategy


Macro-F1 is primary because it weights every class equally. Accuracy is secondary. Five-fold stratified CV estimates development performance; a separate stratified 20% test set is used only for final evaluation.


### 2.4 Baseline training, evaluation, held-out test set, and leakage prevention


In [ ]:
# Split once before model development so the held-out test data is never used in CV.
x_train, x_test, y_train, y_test = train_test_split(
    data[features], data[target], test_size=0.20, stratify=data[target], random_state=SEED
)

# Use five stratified folds to estimate baseline macro-F1 on training data only.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
baseline_model = a1.classifier()
cv_macro_f1 = cross_val_score(baseline_model, x_train, y_train, scoring="f1_macro", cv=cv, n_jobs=1)

# Fit the fixed baseline once on all training rows, then evaluate once on held-out test rows.
baseline_model.fit(x_train, y_train)
baseline_prediction = baseline_model.predict(x_test)
baseline_result = {
    "macro_f1": f1_score(y_test, baseline_prediction, average="macro"),
    "accuracy": accuracy_score(y_test, baseline_prediction),
    "prediction": baseline_prediction,
}
print("CV macro-F1 mean:", cv_macro_f1.mean())
print("CV macro-F1 standard deviation:", cv_macro_f1.std())
print("Held-out macro-F1:", baseline_result["macro_f1"])
print("Held-out accuracy:", baseline_result["accuracy"])


## Task 3 — Generative model development and justification


### 3.1 Selected generative model and justification


I use a conditional variational autoencoder (CVAE). It models continuous tabular features and conditions generation on the class label, allowing targeted minority-class augmentation.


### 3.2 Model architecture, components, and their roles


The CVAE uses a four-value class embedding, an encoder with 64 and 32 ReLU units, an eight-dimensional Gaussian latent space, and a decoder with 32 and 64 ReLU units. The encoder produces latent mean and log variance; the decoder reconstructs features from a latent sample and class embedding.


### 3.3 Tabular-data representation and generation


Each row is represented as ten scaled continuous features plus its class label. During generation, a random latent vector is concatenated with the requested class embedding and decoded back to a synthetic feature row.


### 3.4 Generative-model preprocessing or transformations


The VAE fits `StandardScaler` on the training features only. Synthetic samples are inverse-transformed before quality checks and classifier training.


### 3.5 Assumptions, limitations, and potential risks


A CVAE may smooth rare patterns, generate unrealistic feature combinations, or have limited diversity. Statistical similarity does not prove privacy or that the synthetic data contains no memorised rows.


## Task 4 — Data synthesis and quality assurance


### 4.1 Generative-model training procedure


In [ ]:
# Train the conditional VAE on training data only; the function fits its scaler inside.
vae_model, vae_scaler = a1.train_vae(x_train, y_train, n_classes=data[target].nunique())
print("CVAE training completed with 100 epochs and batch size 256.")


### 4.2 Generation strategy and justification


The generator creates only enough minority-class samples to match the largest training class. This targets imbalance without unnecessarily multiplying the majority class.


### 4.3 Generate the synthetic dataset


In [ ]:
# Sample labels for under-represented classes, generate rows, and save them for inspection.
synthetic_x, synthetic_y = a1.generate_balanced(vae_model, vae_scaler, x_train, y_train)
synthetic_data = synthetic_x.assign(Class=synthetic_y)
synthetic_data.to_csv(ROOT / "data" / "synthetic_train.csv", index=False)
print("Synthetic rows generated:", len(synthetic_data))
display(synthetic_data.head())


### 4.4 Visual and statistical comparison of real and synthetic data


In [ ]:
# Compare representative distributions, correlations, and nearest-neighbour distances.
correlation_gap, real_neighbour_distance, synthetic_neighbour_distance = a1.save_quality_figures(
    x_train, synthetic_x, features
)
display(Image(filename=FIGURES / "real_vs_synthetic.png"))
display(Image(filename=FIGURES / "nearest_neighbour_distances.png"))
print("Mean absolute correlation gap:", correlation_gap)
print("Median real nearest-neighbour distance:", real_neighbour_distance)
print("Median synthetic-to-real nearest-neighbour distance:", synthetic_neighbour_distance)


### 4.5 Critical evaluation of synthetic-data quality


Interpret the two figures and three statistics after execution. Small distribution and correlation differences support similarity, while larger synthetic-to-real nearest-neighbour distances suggest the generated rows are not direct copies. These checks remain limited and do not establish privacy.


## Task 5 — Classification and performance analysis with synthetic data


### 5.1 Train the augmented classifier with the Task 2 model, preprocessing, and hyperparameters


In [ ]:
# Reuse exactly the same random-forest constructor and settings from Task 2.
augmented_model = a1.classifier()
x_augmented = pd.concat([x_train, synthetic_x], ignore_index=True)
y_augmented = pd.concat([y_train, synthetic_y], ignore_index=True)
augmented_model.fit(x_augmented, y_augmented)
augmented_prediction = augmented_model.predict(x_test)
augmented_result = {
    "macro_f1": f1_score(y_test, augmented_prediction, average="macro"),
    "accuracy": accuracy_score(y_test, augmented_prediction),
    "prediction": augmented_prediction,
}
print("Augmented held-out macro-F1:", augmented_result["macro_f1"])
print("Augmented held-out accuracy:", augmented_result["accuracy"])


### 5.2 Compare baseline and augmented classifiers on the same held-out test set


In [ ]:
# Put both held-out results in one compact table and save the comparison figure.
comparison = pd.DataFrame([
    {"model": "Baseline random forest", **{key: value for key, value in baseline_result.items() if key != "prediction"}},
    {"model": "Augmented random forest", **{key: value for key, value in augmented_result.items() if key != "prediction"}},
])
display(comparison)
comparison.to_csv(ROOT / "data" / "performance.csv", index=False)
ax = comparison.set_index("model")[["macro_f1", "accuracy"]].plot.bar(ylim=(0, 1), rot=0)
ax.set_ylabel("score")
ax.set_title("Held-out classifier performance")
plt.tight_layout()
plt.savefig(FIGURES / "performance_comparison.png", dpi=160)
plt.show()


### 5.3 Class-level analysis


In [ ]:
# Report precision, recall, and F1 for every class on the same held-out test set.
print("Baseline classification report")
print(classification_report(y_test, baseline_result["prediction"], zero_division=0))
print("Augmented classification report")
print(classification_report(y_test, augmented_result["prediction"], zero_division=0))


### 5.4 Critical discussion of the synthetic-data effect


Use the held-out comparison and class reports after execution. If macro-F1 improves, identify which classes improved; if it falls or remains similar, explain that imperfect synthetic quality or duplicated class patterns may not add useful decision boundaries.


## Task 6 — Hidden test


### 6.1 Run the selected final model on a new CSV and create Class predictions


In [ ]:
# Select the model by held-out macro-F1, then refit it on all released real data.
selected_name = max(
    [("Baseline random forest", baseline_result), ("Augmented random forest", augmented_result)],
    key=lambda item: item[1]["macro_f1"],
)[0]
final_model = a1.classifier()
if selected_name == "Augmented random forest":
    final_model.fit(
        pd.concat([data[features], synthetic_x], ignore_index=True),
        pd.concat([data[target], synthetic_y], ignore_index=True),
    )
else:
    final_model.fit(data[features], data[target])

# Save the final estimator and ordered feature list for predict.py.
model_path = MODELS / "1173808_Assignment1_final.joblib"
joblib.dump({"model": final_model, "features": features, "selected": selected_name}, model_path)
assert model_path.exists() and model_path.stat().st_size > 0
print("Selected final model:", selected_name)
print("Saved model:", model_path)


Run from the repository root:

```text
python predict.py data/new_beans.csv data/predictions.csv
```

The input must contain the same ten feature columns. The output has one `Class` column.
